# UCU Tuition Reconciliation: Synthetic Data Analysis

This notebook analyses the prototype's fictional student, tuition payment, and Moodle snapshots. It uses the same policy and reconciliation classes as the Streamlit application. No university or personal student data is used. This analysis deliberately excludes non-tuition fees, accommodation, and late penalties. Eligibility depends on the selected tuition milestone; analytical summaries do not define policy.

In [ ]:
from datetime import date
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from src.analytics import ReconciliationAnalytics
from src.policies import ucu_2026_rules
from src.sample_data import load_demo_data
from src.service import FeeReconciliationService
from src.storage import SQLiteRepository

## Load a reproducible snapshot

The tuition-only sample includes confirmed and pending payments, incomplete records, disabled and active accounts, plus orphaned source rows. Use October 31, 2026 to examine the final tuition cutoff. Requirements accumulate and remain active after their cutoffs. Automatic activation is disabled in this analysis run so the notebook does not mutate the sample accounts.

In [ ]:
temporary_directory = TemporaryDirectory()
database_path = Path(temporary_directory.name) / "analysis.sqlite3"
repository = SQLiteRepository(database_path)
load_demo_data(repository)
service = FeeReconciliationService(repository, ucu_2026_rules())
as_of_date = date(2026, 10, 31)
cases = service.reconcile(as_of_date, auto_activate=False)
cases[["student_id", "tuition_due_ugx", "tuition_paid_ugx", "tuition_balance_ugx", "tuition_paid_percent", "eligible", "case_status"]]

## Reconciliation outcomes and data quality

In [ ]:
analytics = ReconciliationAnalytics(repository)
summary = analytics.summary(cases)
summary

In [ ]:
outcomes = cases.groupby("case_status", dropna=False).size().sort_values(ascending=False)
outcomes

In [ ]:
quality_findings = service.data_quality()
quality_findings

## Payment coverage distribution

This descriptive distribution is not an eligibility decision. The policy checks confirmed payments against assessed tuition thresholds; non-tuition charges are outside this prototype's scope.

In [ ]:
coverage = cases["tuition_paid_percent"].dropna().to_numpy(dtype=float)
coverage_statistics = pd.Series(
    {
        "students_with_assessed_tuition": int(coverage.size),
        "mean_coverage_percent": float(np.mean(coverage)) if coverage.size else 0.0,
        "median_coverage_percent": float(np.median(coverage)) if coverage.size else 0.0,
        "minimum_coverage_percent": float(np.min(coverage)) if coverage.size else 0.0,
        "maximum_coverage_percent": float(np.max(coverage)) if coverage.size else 0.0,
    }
)
coverage_statistics

## Policy boundary check

November 1 is after the schedule's final tuition cutoff. The engine continues to require full assessed tuition; students can still qualify after paying late. The system does not add a new deadline.

In [ ]:
after_window = service.reconcile(date(2026, 11, 1), auto_activate=False)
after_window[["student_id", "rule_name", "eligible", "case_status"]]

## Interpretation

The sample is intentionally synthetic and small, so the summaries demonstrate system behavior rather than estimate real university performance. Confirmed tuition payments alone contribute to fee coverage. Unmatched records and missing tuition assessments are surfaced for review. Frank Ouma (UCU-2026-006) has a cleared tuition balance and demonstrates simulated activation in the Streamlit reconciliation. Actual payment-to-reactivation turnaround analysis requires trusted source timestamps and authorized production integrations.